# Module 01 — Notebook 01: Gradient Methods on CNN

## Learning Objectives
By completing this notebook, you will:
1. Apply all four gradient-based methods (Saliency, Input×Gradient, Guided Backprop, Deconvolution) to a pretrained ResNet
2. Produce side-by-side attribution visualizations for multiple images
3. Quantitatively compare attribution maps using correlation and sparsity metrics
4. Run the randomization sanity check to identify architecture-dependent methods

## Prerequisites
- Module 00 notebooks completed
- Guides 01 and 02 from Module 01 read

## Estimated Time: 15 minutes

---

## Setup

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import copy
import urllib.request
import io
import json
from PIL import Image

from captum.attr import (
    Saliency, InputXGradient, GuidedBackprop, Deconvolution
)
from captum.attr import visualization as viz

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Standard ImageNet preprocessing
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

def to_display(tensor):
    """Convert (1,C,H,W) tensor to (H,W,C) numpy for display."""
    return denormalize(tensor.squeeze(0)).permute(1,2,0).numpy()

def to_attr_np(tensor):
    """Convert (1,C,H,W) attribution tensor to (H,W,C) numpy."""
    return tensor.squeeze(0).cpu().permute(1,2,0).detach().numpy()

## 1. Load Model and Images

We use ResNet-50 pretrained on ImageNet and three sample images covering different subject types: a dog, a bird, and a castle. Having three images lets us verify that attribution patterns are input-dependent (not constant).

In [ ]:
# Load pretrained ResNet-50
model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()  # CRITICAL: must be in eval mode
print(f"ResNet-50 loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load ImageNet labels
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[207] = "golden retriever"; labels[208] = "Labrador retriever"
    labels[16] = "robin"; labels[483] = "castle"; labels[281] = "tabby cat"

# Sample images (public domain from Wikimedia Commons)
IMAGE_URLS = [
    ("Labrador dog",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"),
    ("Tabby cat",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"),
    ("Robin bird",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b8/2009-04-21_-_Redditch_-_Robins_in_the_garden_-_juvenile_2.jpg/320px-2009-04-21_-_Redditch_-_Robins_in_the_garden_-_juvenile_2.jpg"),
]

def load_image(url):
    """Load image from URL, return PIL Image."""
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            return Image.open(io.BytesIO(resp.read())).convert('RGB')
    except Exception:
        # Fallback: solid color image for pipeline testing
        img = np.random.randint(80, 180, (320, 320, 3), dtype=np.uint8)
        return Image.fromarray(img)

# Load and preprocess all images
images_raw = []
images_tensor = []
images_display = []

for name, url in IMAGE_URLS:
    print(f"Loading: {name}...")
    raw = load_image(url)
    tensor = preprocess(raw).unsqueeze(0).to(DEVICE)
    images_raw.append(raw)
    images_tensor.append(tensor)
    images_display.append(to_display(tensor))

# Get predictions
predictions = []
with torch.no_grad():
    for i, (name, _) in enumerate(IMAGE_URLS):
        probs = torch.softmax(model(images_tensor[i]), dim=1)
        top_class = probs.argmax().item()
        top_prob = probs.max().item()
        predictions.append((top_class, top_prob))
        print(f"  {name} → {labels[top_class]} ({top_prob:.1%})")

# Show all three images
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, ((name, _), disp, (cls, prob)) in enumerate(
        zip(IMAGE_URLS, images_display, predictions)):
    axes[i].imshow(disp)
    axes[i].set_title(f"{name}\n→ {labels[cls]}\n({prob:.1%})", fontsize=10)
    axes[i].axis('off')
plt.suptitle("Input Images and ResNet-50 Predictions", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Compute All Four Gradient Attributions

We apply all four gradient methods to the first image. The computation is structured as a dictionary to enable easy comparison.

In [ ]:
def compute_gradient_attributions(model, input_tensor, target_class):
    """
    Compute all four gradient-based attributions for one image.

    Parameters
    ----------
    model : nn.Module in eval() mode
    input_tensor : (1, C, H, W) tensor
    target_class : int — class to explain

    Returns
    -------
    dict {name: (1, C, H, W) attribution tensor}
    """
    # Method dictionary — parameterize by method name
    method_classes = {
        'Saliency':         Saliency,
        'Input×Gradient':   InputXGradient,
        'Guided Backprop':  GuidedBackprop,
        'Deconvolution':    Deconvolution,
    }

    attributions = {}
    for name, MethodClass in method_classes.items():
        # Fresh clone with gradients enabled for each method
        inp = input_tensor.clone().requires_grad_(True)
        method = MethodClass(model)
        attr = method.attribute(inp, target=target_class)
        # Detach to release gradient graph and save memory
        attributions[name] = attr.detach().cpu()
        print(f"  {name}: shape={attr.shape}, "
              f"range=[{attr.min():.3f}, {attr.max():.3f}]")

    return attributions


def normalize_attr_for_display(attr_tensor, percentile=99):
    """
    Normalize attribution magnitude to [0, 1] for display.
    Uses percentile clipping to handle outliers.
    """
    # Take mean absolute value across channels → (H, W)
    mag = attr_tensor.squeeze(0).permute(1, 2, 0).abs().mean(-1).numpy()
    vmax = np.percentile(mag, percentile)
    return np.clip(mag, 0, vmax) / (vmax + 1e-8)


# Compute for the first image (Labrador dog)
target_class, target_prob = predictions[0]
print(f"Computing attributions for '{labels[target_class]}' ({target_prob:.1%})...")

grad_attrs = compute_gradient_attributions(
    model, images_tensor[0].cpu(), target_class
)
print(f"\nAll attributions computed.")

## 3. Side-by-Side Visualization

The central visualization: four methods × three display modes (heatmap, overlay, distribution). This grid lets you immediately see the qualitative differences between methods.

In [ ]:
def plot_four_methods(attributions, image_display, predicted_class_name):
    """
    Create a grid visualization: methods × display modes.

    Rows: one per attribution method
    Columns: [heatmap, overlay on image, attribution distribution]
    """
    method_names = list(attributions.keys())
    n = len(method_names)

    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))

    for row, name in enumerate(method_names):
        attr = attributions[name]
        attr_norm = normalize_attr_for_display(attr)
        attr_raw  = attr.squeeze(0).permute(1, 2, 0).numpy()  # Signed values

        # Column 0: Attribution heatmap
        im0 = axes[row, 0].imshow(attr_norm, cmap='hot', vmin=0, vmax=1)
        axes[row, 0].set_title(f"{name}\nHeatmap", fontsize=10)
        axes[row, 0].axis('off')
        plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)

        # Column 1: Attribution overlaid on image
        axes[row, 1].imshow(image_display)
        axes[row, 1].imshow(
            attr_norm, alpha=0.65, cmap='hot',
            vmin=np.percentile(attr_norm, 80)
        )
        axes[row, 1].set_title(f"{name}\nOverlay", fontsize=10)
        axes[row, 1].axis('off')

        # Column 2: Distribution of raw attribution values
        flat = attr_raw.flatten()
        axes[row, 2].hist(flat, bins=80, color='steelblue', alpha=0.75, edgecolor='none')
        axes[row, 2].axvline(0, color='red', lw=1, linestyle='--', label='zero')
        axes[row, 2].set_title(f"{name}\nValue Distribution", fontsize=10)
        axes[row, 2].set_xlabel('Attribution value')
        axes[row, 2].set_ylabel('Pixel count')
        axes[row, 2].set_yscale('log')
        axes[row, 2].legend(fontsize=8)

    plt.suptitle(
        f"Gradient Attribution Methods — Predicted: '{predicted_class_name}'",
        fontsize=14, y=1.01
    )
    plt.tight_layout()
    plt.show()


plot_four_methods(
    grad_attrs,
    images_display[0],
    labels[target_class]
)

## 4. Quantitative Comparison

Visual comparison is useful but subjective. We add three quantitative metrics:

1. **Sparsity:** What fraction of pixels carry 80% of the total attribution? Sparser = more focused.
2. **Pairwise correlation:** How similar are the attribution maps between methods?
3. **Signal-to-noise ratio:** How much does attribution concentrate on the top-10% of pixels vs. the rest?

In [ ]:
def compute_attribution_metrics(attributions):
    """
    Compute quantitative metrics for each attribution method.

    Metrics:
    - sparsity: fraction of pixels needed to account for 80% of total attribution
    - snr: std(top 10%) / std(bottom 90%)
    - range: max - min of attribution magnitude
    """
    metrics = {}
    vectors = {}  # Flat vectors for pairwise correlation

    for name, attr in attributions.items():
        mag = attr.squeeze(0).permute(1, 2, 0).abs().mean(-1).numpy().flatten()
        vectors[name] = mag

        # Sort descending
        sorted_mag = np.sort(mag)[::-1]
        cumsum = np.cumsum(sorted_mag) / (sorted_mag.sum() + 1e-8)
        # Sparsity: fraction of pixels to reach 80% cumulative attribution
        sparsity_80 = np.searchsorted(cumsum, 0.80) / len(mag)

        # SNR
        threshold = np.percentile(mag, 90)
        top = mag[mag >= threshold]
        bot = mag[mag < threshold]
        snr = top.std() / (bot.std() + 1e-8)

        metrics[name] = {
            'sparsity_80pct': sparsity_80,
            'snr': snr,
            'range': mag.max() - mag.min(),
            'mean': mag.mean(),
        }

    return metrics, vectors


metrics, vectors = compute_attribution_metrics(grad_attrs)

# Print metrics table
print("Attribution Metrics")
print("=" * 70)
print(f"{'Method':<22} {'Sparsity (80%)':<16} {'SNR':<10} {'Range':<10}")
print("-" * 70)
for name, m in metrics.items():
    print(f"{name:<22} {m['sparsity_80pct']:<16.3f} {m['snr']:<10.2f} {m['range']:<10.4f}")

print("\nSparsity (80%): fraction of pixels accounting for 80% of attribution.")
print("  Lower = more focused attribution (fewer pixels carry most information).")
print("SNR: std(top 10%) / std(bottom 90%). Higher = clearer signal vs. background.")

# Pairwise correlation matrix
method_names = list(vectors.keys())
n = len(method_names)
corr_matrix = np.zeros((n, n))
for i, n1 in enumerate(method_names):
    for j, n2 in enumerate(method_names):
        corr_matrix[i, j] = np.corrcoef(vectors[n1], vectors[n2])[0, 1]

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(method_names, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(method_names, fontsize=9)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha='center', va='center',
                fontsize=10, color='black')
ax.set_title("Attribution Map Correlation (pairwise)\n"
             "High correlation = methods agree on which regions matter")
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("Correlation > 0.7: methods largely agree on important regions.")
print("Correlation < 0.3: methods focus on very different features.")

## 5. The Sanity Check: Randomization Test

This is the most important experiment in this module. We compare attributions from the trained ResNet-50 vs. a randomly initialized model with the same architecture.

**Expected result:**
- Saliency and Input×Gradient: low correlation (attributions change significantly)
- Guided Backprop and Deconvolution: high correlation (attributions barely change)

This demonstrates that Guided Backprop reflects the architecture, not the learned weights.

In [ ]:
# Create a randomly initialized ResNet-50 (same architecture, random weights)
random_model = copy.deepcopy(model)
for module in random_model.modules():
    if hasattr(module, 'reset_parameters'):
        module.reset_parameters()
random_model.eval()

print("Random model created (same architecture, random weights).")

# Get the "prediction" for the random model (top class from random outputs)
# We use the same target class as before for a fair comparison
print(f"\nComputing attributions for target class: {labels[target_class]}")
print("Trained model attributions (already computed above)...")
print("Random model attributions...")

# Compute attributions for the random model
random_attrs = compute_gradient_attributions(
    random_model, images_tensor[0].cpu(), target_class
)

# Compare trained vs random for each method
print("\n" + "=" * 65)
print("SANITY CHECK: Trained vs Random Model Correlation")
print("=" * 65)
print(f"{'Method':<22} {'Corr (trained vs random)':<30} {'Verdict'}")
print("-" * 65)

for name in grad_attrs:
    v_trained = grad_attrs[name].squeeze(0).permute(1,2,0).abs().mean(-1).numpy().flatten()
    v_random  = random_attrs[name].squeeze(0).permute(1,2,0).abs().mean(-1).numpy().flatten()
    corr = np.corrcoef(v_trained, v_random)[0, 1]
    verdict = "ARCHITECTURE-DEPENDENT" if corr > 0.4 else "Faithful"
    print(f"{name:<22} {corr:<30.3f} {verdict}")

print("\nInterpretation:")
print("  Low correlation (< 0.4): attribution reflects LEARNED WEIGHTS (good)")
print("  High correlation (> 0.4): attribution reflects ARCHITECTURE (bad — not faithful)")

# Visual comparison for GuidedBackprop specifically
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.suptitle("Sanity Check: Trained vs Random Model Attributions", fontsize=13)

for col_idx, method_name in enumerate(["Saliency", "Input×Gradient", "Guided Backprop"]):
    trained_norm = normalize_attr_for_display(grad_attrs[method_name])
    random_norm  = normalize_attr_for_display(random_attrs[method_name])

    axes[0, col_idx].imshow(trained_norm, cmap='hot', vmin=0, vmax=1)
    axes[0, col_idx].set_title(f"{method_name}\n(Trained Model)", fontsize=10)
    axes[0, col_idx].axis('off')

    axes[1, col_idx].imshow(random_norm, cmap='hot', vmin=0, vmax=1)
    axes[1, col_idx].set_title(f"{method_name}\n(Random Model)", fontsize=10)
    axes[1, col_idx].axis('off')

plt.tight_layout()
plt.show()

print("\nObservation:")
print("Saliency and Input×Gradient: Maps should look DIFFERENT (random model has noise-like maps)")
print("Guided Backprop: Maps may look SIMILAR — this is the failure.")
print("This demonstrates that Guided Backprop reflects architecture, not learned features.")

## 6. Multi-Image Attribution Gallery

Verify that attributions are input-dependent by applying Saliency and Input×Gradient to all three images. Each image should produce a distinct, image-specific attribution pattern.

In [ ]:
# Apply Saliency and Input×Gradient to all three images
print("Computing attributions for all three images...")

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
fig.suptitle(
    "Gradient Attributions Across Three Images\n"
    "(Shows that attributions are input-specific)",
    fontsize=13
)

col_titles = ["Original Image", "Saliency", "Input×Gradient", "Overlay (Saliency)"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight='bold')

for row, (img_name_url, tensor, disp, (pred_class, pred_prob)) in enumerate(
        zip(IMAGE_URLS, images_tensor, images_display, predictions)):

    img_name = img_name_url[0]
    pred_label = labels[pred_class]

    # Compute saliency
    inp_sal = tensor.cpu().clone().requires_grad_(True)
    sal_attr = Saliency(model).attribute(inp_sal, target=pred_class).detach().cpu()
    sal_norm = normalize_attr_for_display(sal_attr)

    # Compute Input×Gradient
    inp_ixg = tensor.cpu().clone().requires_grad_(True)
    ixg_attr = InputXGradient(model).attribute(inp_ixg, target=pred_class).detach().cpu()
    ixg_norm = normalize_attr_for_display(ixg_attr)

    # Column 0: Original image
    axes[row, 0].imshow(disp)
    axes[row, 0].set_ylabel(f"{img_name}\n→ {pred_label}\n({pred_prob:.1%})",
                             fontsize=9, rotation=0, ha='right', va='center')
    axes[row, 0].axis('off')

    # Column 1: Saliency heatmap
    axes[row, 1].imshow(sal_norm, cmap='hot', vmin=0, vmax=1)
    axes[row, 1].axis('off')

    # Column 2: Input×Gradient heatmap
    axes[row, 2].imshow(ixg_norm, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].axis('off')

    # Column 3: Saliency overlaid on image
    axes[row, 3].imshow(disp)
    axes[row, 3].imshow(sal_norm, alpha=0.6, cmap='hot',
                         vmin=np.percentile(sal_norm, 80))
    axes[row, 3].axis('off')

    print(f"  Row {row+1}: {img_name} → {pred_label}")

plt.tight_layout()
plt.show()

print("\nEach image produces a distinct attribution pattern.")
print("Attribution reflects the specific image content, not a fixed template.")

## Summary

### What You Observed

1. **All four gradient methods** produce attribution maps in the same shape as the input
2. **Visual differences** are significant: Saliency is noisy, Input×Gradient adds input weighting, Guided Backprop produces cleaner but architecture-dependent outputs
3. **Quantitatively:** methods agree moderately (correlation 0.4-0.7 typically) but differ in sparsity and signal-to-noise ratio
4. **Sanity check:** Guided Backprop (and Deconvolution) show high correlation between trained and randomly initialized models — they reflect the architecture, not the learned weights
5. **Attributions are input-specific:** the same method produces different maps for different images

### Key Conclusions

- Use **Saliency or Input×Gradient** for fast debugging (faithful to model, fast)
- Use **Guided Backprop** only for clean presentation visuals (not for validation)
- **Always run the randomization sanity check** on any new attribution method

### What's Next

**Notebook 02:** Apply gradient methods to tabular neural networks — different input modality, same API.

**Notebook 03:** Deep dive into saliency noise and why Integrated Gradients (Module 02) is a better choice for rigorous attribution.